In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("orders.csv")
payments = pd.read_csv("payments.csv")

In [3]:
orders.head()

,order_id,customer_id,order_amount,order_status
0,ORD1001,C101,2500,COMPLETED
1,ORD1002,C102,1800,COMPLETED
2,ORD1003,C103,3200,COMPLETED
3,ORD1004,C104,950,CANCELLED
4,ORD1005,C105,4100,COMPLETED


In [4]:
payments.head()

,payment_id,order_id,payment_amount,payment_status
0,PAY501,ORD1001,2500,SUCCESS
1,PAY502,ORD1002,1800,SUCCESS
2,PAY503,ORD1003,3000,SUCCESS
3,PAY504,ORD1004,950,REFUNDED
4,PAY505,ORD1005,4100,SUCCESS


### Inspection

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   order_id      8 non-null      str  
 1   customer_id   8 non-null      str  
 2   order_amount  8 non-null      int64
 3   order_status  8 non-null      str  
dtypes: int64(1), str(3)
memory usage: 388.0 bytes


In [6]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   payment_id      7 non-null      str  
 1   order_id        7 non-null      str  
 2   payment_amount  7 non-null      int64
 3   payment_status  7 non-null      str  
dtypes: int64(1), str(3)
memory usage: 356.0 bytes


In [7]:
orders[orders.duplicated()]

,order_id,customer_id,order_amount,order_status
7,ORD1007,C107,2200,COMPLETED


In [8]:
orders[orders['order_id'] == 'ORD1007']

,order_id,customer_id,order_amount,order_status
6,ORD1007,C107,2200,COMPLETED
7,ORD1007,C107,2200,COMPLETED


Same duplicate records, so dropping one

In [9]:
orders = orders.drop_duplicates()

In [10]:
orders

,order_id,customer_id,order_amount,order_status
0,ORD1001,C101,2500,COMPLETED
1,ORD1002,C102,1800,COMPLETED
2,ORD1003,C103,3200,COMPLETED
3,ORD1004,C104,950,CANCELLED
4,ORD1005,C105,4100,COMPLETED
5,ORD1006,C106,1500,COMPLETED
6,ORD1007,C107,2200,COMPLETED


In [11]:
payments[payments.duplicated()]

,payment_id,order_id,payment_amount,payment_status


Reconcile orders against payments while preserving records that exist in only one system \
Validate the expected merge relationship so unexpected duplicate keys don't silently produce incorrect reconciliation results.

In [12]:
all_data = pd.merge(
    left=orders,
    right=payments,
    how="outer",
    on="order_id",
    validate="1:1",
    indicator=True
)

In [13]:
all_data

,order_id,customer_id,order_amount,order_status,payment_id,payment_amount,payment_status,_merge
0,ORD1001,C101,2500.0,COMPLETED,PAY501,2500.0,SUCCESS,both
1,ORD1002,C102,1800.0,COMPLETED,PAY502,1800.0,SUCCESS,both
2,ORD1003,C103,3200.0,COMPLETED,PAY503,3000.0,SUCCESS,both
3,ORD1004,C104,950.0,CANCELLED,PAY504,950.0,REFUNDED,both
4,ORD1005,C105,4100.0,COMPLETED,PAY505,4100.0,SUCCESS,both
5,ORD1006,C106,1500.0,COMPLETED,PAY507,1500.0,FAILED,both
6,ORD1007,C107,2200.0,COMPLETED,NaN,NaN,NaN,left_only
7,ORD1008,NaN,NaN,NaN,PAY506,1750.0,SUCCESS,right_only


Classify the reconciliation result into these business situations:
- MATCHED
- AMOUNT_MISMATCH
- ORDER_WITHOUT_PAYMENT
- ORPHAN_PAYMENT

In [14]:
all_data.loc[
    all_data['_merge'] == 'left_only',
    ['recon_status']
] = 'ORDER_WITHOUT_PAYMENT'

all_data.loc[
    all_data['_merge'] == 'right_only',
    ['recon_status']
] = 'ORPHAN_PAYMENT'

all_data.loc[
    (all_data['_merge'] == 'both') &
    (all_data['order_amount'] != all_data['payment_amount']),
    ['recon_status']
] = 'AMOUNT_MISMATCH'

all_data.loc[
    (all_data['_merge'] == 'both') &
    (all_data['order_amount'] == all_data['payment_amount']),
    ['recon_status']
] = 'MATCHED'

In [15]:
all_data

,order_id,customer_id,order_amount,order_status,payment_id,payment_amount,payment_status,_merge,recon_status
0,ORD1001,C101,2500.0,COMPLETED,PAY501,2500.0,SUCCESS,both,MATCHED
1,ORD1002,C102,1800.0,COMPLETED,PAY502,1800.0,SUCCESS,both,MATCHED
2,ORD1003,C103,3200.0,COMPLETED,PAY503,3000.0,SUCCESS,both,AMOUNT_MISMATCH
3,ORD1004,C104,950.0,CANCELLED,PAY504,950.0,REFUNDED,both,MATCHED
4,ORD1005,C105,4100.0,COMPLETED,PAY505,4100.0,SUCCESS,both,MATCHED
5,ORD1006,C106,1500.0,COMPLETED,PAY507,1500.0,FAILED,both,MATCHED
6,ORD1007,C107,2200.0,COMPLETED,NaN,NaN,NaN,left_only,ORDER_WITHOUT_PAYMENT
7,ORD1008,NaN,NaN,NaN,PAY506,1750.0,SUCCESS,right_only,ORPHAN_PAYMENT


Identify the completed order(s) without a successful payment. Think carefully here: a payment record existing doesn't necessarily mean the customer successfully paid.

In [16]:
all_data.loc[
    (all_data['order_status'] == 'COMPLETED') &
    (all_data['payment_status'] != 'SUCCESS'),
    :
]

,order_id,customer_id,order_amount,order_status,payment_id,payment_amount,payment_status,_merge,recon_status
5,ORD1006,C106,1500.0,COMPLETED,PAY507,1500.0,FAILED,both,MATCHED
6,ORD1007,C107,2200.0,COMPLETED,NaN,NaN,NaN,left_only,ORDER_WITHOUT_PAYMENT


Identify any successful payment that has no corresponding order.

In [17]:
all_data[all_data['recon_status'] == 'ORPHAN_PAYMENT']

,order_id,customer_id,order_amount,order_status,payment_id,payment_amount,payment_status,_merge,recon_status
7,ORD1008,NaN,NaN,NaN,PAY506,1750.0,SUCCESS,right_only,ORPHAN_PAYMENT


Calculate:
- Total COMPLETED order amount
- Total SUCCESS payment amount
- Difference

In [18]:
all_data

,order_id,customer_id,order_amount,order_status,payment_id,payment_amount,payment_status,_merge,recon_status
0,ORD1001,C101,2500.0,COMPLETED,PAY501,2500.0,SUCCESS,both,MATCHED
1,ORD1002,C102,1800.0,COMPLETED,PAY502,1800.0,SUCCESS,both,MATCHED
2,ORD1003,C103,3200.0,COMPLETED,PAY503,3000.0,SUCCESS,both,AMOUNT_MISMATCH
3,ORD1004,C104,950.0,CANCELLED,PAY504,950.0,REFUNDED,both,MATCHED
4,ORD1005,C105,4100.0,COMPLETED,PAY505,4100.0,SUCCESS,both,MATCHED
5,ORD1006,C106,1500.0,COMPLETED,PAY507,1500.0,FAILED,both,MATCHED
6,ORD1007,C107,2200.0,COMPLETED,NaN,NaN,NaN,left_only,ORDER_WITHOUT_PAYMENT
7,ORD1008,NaN,NaN,NaN,PAY506,1750.0,SUCCESS,right_only,ORPHAN_PAYMENT


In [19]:
group_by_order_status = all_data.groupby('order_status', as_index=False).agg(
    total_order_amt=('order_amount', 'sum')
)

completed_order_amount = group_by_order_status[group_by_order_status['order_status'] == 'COMPLETED']
completed_order_amount

,order_status,total_order_amt
1,COMPLETED,15300.0


In [20]:
group_by_payment_status = all_data.groupby('payment_status', as_index=False).agg(
    total_payment_amt=('payment_amount', sum)
)

success_payment_amount = group_by_payment_status[group_by_payment_status['payment_status'] == 'SUCCESS']
success_payment_amount

,payment_status,total_payment_amt
2,SUCCESS,13150.0


In [21]:
difference = completed_order_amount['total_order_amt'].sum() - success_payment_amount['total_payment_amt'].sum()

In [22]:
difference

np.float64(2150.0)